# Day3_07 · 미니 프로젝트

건축·건설재료·환경GIS 자료를 확인하고, 한 경로를 골라 결과를 세 문장으로 정리합니다.

## 이 노트북에서 확인할 내용

- 전공과 연결된 분석 질문을 선택할 수 있습니다.
- 건물·콘크리트·기상·지도 자료에서 표와 그래프를 만들 수 있습니다.
- 관찰한 결과와 해석의 한계를 구분해 발표할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 실행하고 바로 아래 결과를 확인합니다. 값을 만드는 셀 다음에는 그 값을 화면에 표시하는 셀이 이어집니다. 수정 안내가 있는 셀에서는 주석 아래의 값만 바꿉니다.

### D3B-07-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-07-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-07-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-07-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-07-05 · 표 도구 불러오기

In [ ]:
import pandas as pd

### D3B-07-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt

### D3B-07-07 · Seaborn 불러오기

In [ ]:
import seaborn as sns

### D3B-07-08 · JSON 도구 불러오기

In [ ]:
import json

### D3B-07-09 · 세 가지 전공 질문 확인

In [ ]:
project_questions = {
    '건축': '건물 높이에 따라 평균 난방·냉방 부하는 어떻게 다른가?',
    '재료': '재령 28일 콘크리트에서 시멘트량과 압축강도는 어떻게 나타나는가?',
    '환경GIS': '울산의 월별 기온·강수와 구군별 기상 격자값은 어떻게 다른가?',
}

In [ ]:
project_questions

## A. 건축공학 · 건물 외피와 냉난방 부하

건물 형상 시뮬레이션 768행을 높이별로 요약하고, 외피 조건과 부하를 그림으로 확인합니다.

### D3B-07-10 · 건물 에너지자료 읽기

In [ ]:
building = pd.read_csv(DATA_DIR / 'building_energy.csv')

In [ ]:
building

### D3B-07-11 · 건물 에너지자료 크기 확인

In [ ]:
building.shape

### D3B-07-12 · 건물 높이별 냉난방 부하 표 만들기

In [ ]:
building_table = building.groupby('overall_height_m', as_index=False)[['heating_load', 'cooling_load']].mean()

In [ ]:
building_table

### D3B-07-13 · 높이별 평균 난방 부하 그래프 만들기

In [ ]:
sns.barplot(data=building_table, x='overall_height_m', y='heating_load')
plt.title('Mean heating load by building height')
plt.show()

### D3B-07-14 · 조밀함과 난방 부하 그래프 만들기

In [ ]:
sns.scatterplot(data=building, x='relative_compactness', y='heating_load')
plt.title('Building compactness and heating load')
plt.show()

### D3B-07-15 · 유리 면적과 냉방 부하 그래프 만들기

In [ ]:
sns.scatterplot(data=building, x='glazing_area_ratio', y='cooling_load')
plt.title('Glazing area and cooling load')
plt.show()

## B. 건설재료 · 콘크리트 배합과 압축강도

재령 28일인 배합만 선택한 뒤 시멘트량과 압축강도의 분포를 확인합니다.

### D3B-07-16 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete

### D3B-07-17 · 콘크리트 강도자료 크기 확인

In [ ]:
concrete.shape

### D3B-07-18 · 재령 28일 배합 선택

In [ ]:
concrete_28day = concrete[concrete['age_days'] == 28]

In [ ]:
concrete_28day

### D3B-07-19 · 재령 28일 배합 수 확인

In [ ]:
concrete_28day.shape

### D3B-07-20 · 시멘트량과 압축강도 그래프 만들기

In [ ]:
sns.scatterplot(data=concrete_28day, x='cement_kg_m3', y='strength_mpa')
plt.title('28-day concrete strength')
plt.show()

### D3B-07-21 · 재령 28일 강도 요약표 만들기

In [ ]:
concrete_table = concrete_28day['strength_mpa'].describe()

In [ ]:
concrete_table

## C. 건설환경 · 울산 기상과 GIS

일별 기상자료를 월별로 요약하고, 울산 구·군 경계 위에 다섯 기상 격자점을 표시합니다.

### D3B-07-22 · 울산 일별 기상자료 읽기

In [ ]:
weather = pd.read_csv(DATA_DIR / 'weather_daily.csv')

In [ ]:
weather

### D3B-07-23 · 날짜를 날짜 형식으로 바꾸기

In [ ]:
weather['date'] = pd.to_datetime(weather['date'])

In [ ]:
weather['date'].head()

### D3B-07-24 · 월을 새 열로 추가

In [ ]:
weather['month'] = weather['date'].dt.month

In [ ]:
weather[['date', 'month']].head()

### D3B-07-25 · 월별 기상 표 만들기

In [ ]:
monthly_weather = weather.groupby('month', as_index=False).agg(
    temperature_mean_c=('temperature_mean_c', 'mean'),
    precipitation_total_mm=('precipitation_mm', 'sum'),
)

In [ ]:
monthly_weather

### D3B-07-26 · 월별 평균기온 그래프 만들기

In [ ]:
sns.lineplot(data=monthly_weather, x='month', y='temperature_mean_c', marker='o')
plt.title('Ulsan monthly mean temperature in 2024')
plt.show()

### D3B-07-27 · 월별 누적강수량 그래프 만들기

In [ ]:
sns.barplot(data=monthly_weather, x='month', y='precipitation_total_mm')
plt.title('Ulsan monthly precipitation in 2024')
plt.show()

### D3B-07-28 · 울산 기상 격자점 읽기

In [ ]:
weather_points = pd.read_csv(DATA_DIR / 'ulsan_weather_points.csv')

In [ ]:
weather_points

### D3B-07-29 · 울산 구·군 경계 읽기

In [ ]:
ulsan_geo = json.loads((DATA_DIR / 'ulsan_districts.geojson').read_text(encoding='utf-8'))

In [ ]:
[feature['properties']['district'] for feature in ulsan_geo['features']]

### D3B-07-30 · 경계 좌표를 꺼내는 함수 준비

In [ ]:
def geometry_rings(geometry):
    if geometry['type'] == 'Polygon':
        return [geometry['coordinates'][0]]
    return [polygon[0] for polygon in geometry['coordinates']]

### D3B-07-31 · 구·군별 연평균기온 지도 만들기

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.08, edgecolor='black')
plt.scatter(weather_points['longitude'], weather_points['latitude'], c=weather_points['temperature_mean_c'], cmap='coolwarm', s=110)
plt.colorbar(label='Mean temperature (C)')
plt.title('2024 Ulsan weather grid points')
plt.show()

### D3B-07-32 · 발표할 전공 경로 정하기

In [ ]:
# 건축, 재료, 환경GIS 중 하나를 적으세요.
selected_track = '건축'

In [ ]:
selected_track

### D3B-07-33 · 선택한 분석 질문 확인

In [ ]:
research_question = project_questions[selected_track]

In [ ]:
research_question

### D3B-07-34 · 결과 문장 1 작성

In [ ]:
# 선택한 표에서 확인한 숫자를 넣으세요.
result_sentence_1 = '결과 표에서 ___는 ___로 확인되었습니다.'

In [ ]:
result_sentence_1

### D3B-07-35 · 결과 문장 2 작성

In [ ]:
# 선택한 그래프에서 보이는 방향이나 차이를 적으세요.
result_sentence_2 = '그래프에서는 ___한 모습이 보였습니다.'

In [ ]:
result_sentence_2

### D3B-07-36 · 결과 문장 3 작성

In [ ]:
# 이 자료만으로 판단할 수 없는 내용을 적으세요.
result_sentence_3 = '이 표와 그래프만으로 차이의 원인을 판단할 수는 없습니다.'

In [ ]:
result_sentence_3

### D3B-07-37 · 발표 카드 만들기

In [ ]:
presentation_card = pd.Series({
    '전공 경로': selected_track,
    '질문': research_question,
    '표에서 확인': result_sentence_1,
    '그래프에서 확인': result_sentence_2,
    '한계': result_sentence_3,
})

In [ ]:
presentation_card

### D3B-07-38 · Codex에 요청할 문장 준비

In [ ]:
codex_request = f'''이 Notebook의 기존 셀은 수정하지 마세요.
제가 선택한 경로는 {selected_track}이고 질문은 {research_question}입니다.
이미 만든 DataFrame만 사용해 질문에 도움이 되는 그래프 한 개를 제안해 주세요.
먼저 사용할 열과 두 축의 뜻을 설명하고, 짧은 코드 셀 하나만 작성해 주세요.
그래프에서 보이는 관계를 원인으로 단정하지 마세요.'''.strip()

In [ ]:
codex_request

## 프로젝트 확장 · 기준값을 바꾸어 결과 확인하기

각 전공 자료에서 숫자 하나만 바꾸고 표 또는 그래프가 어떻게 달라지는지 확인합니다.

### D3B-07-39 · 건물 난방 부하 기준 정하기

In [ ]:
# 25를 20 또는 30으로 바꿔 보세요.
heating_limit = 25

In [ ]:
heating_limit

### D3B-07-40 · 난방 부하가 기준 이상인 건물 찾기

In [ ]:
high_heating_buildings = building[building['heating_load'] >= heating_limit]

In [ ]:
high_heating_buildings.head(10)

### D3B-07-41 · 기준 이상 건물 수 확인

In [ ]:
high_heating_buildings.shape

### D3B-07-42 · 콘크리트 강도 기준 정하기

In [ ]:
# MPa 단위입니다. 40을 30 또는 50으로 바꿔 보세요.
strength_limit = 40

In [ ]:
strength_limit

### D3B-07-43 · 기준 이상 콘크리트 배합 찾기

In [ ]:
strong_concrete = concrete[concrete['strength_mpa'] >= strength_limit]

In [ ]:
strong_concrete.head(10)

### D3B-07-44 · 기준 이상 배합 수 확인

In [ ]:
strong_concrete.shape

### D3B-07-45 · 기상 최고기온 기준 정하기

In [ ]:
# 섭씨 30도를 28 또는 33으로 바꿔 보세요.
hot_limit = 30

In [ ]:
hot_limit

### D3B-07-46 · 기준 이상인 날 찾기

In [ ]:
hot_weather = weather[weather['temperature_max_c'] >= hot_limit]

In [ ]:
hot_weather[['date', 'temperature_max_c']].head(10)

### D3B-07-47 · 기준 이상인 날 수 확인

In [ ]:
hot_weather.shape

### D3B-07-48 · 프로젝트 완료 확인표 만들기

In [ ]:
project_check = pd.Series({
    '건물 결과 표 확인': len(building_table) > 0,
    '콘크리트 결과 표 확인': len(concrete_28day) > 0,
    '월별 기상 표 확인': len(monthly_weather) == 12,
    '울산 구군 지도 확인': len(weather_points) == 5,
    '결과 문장 수': 3,
})

In [ ]:
project_check

### D3B-07-49 · Codex 답변 확인 기준 만들기

In [ ]:
codex_check = pd.Series({
    '기존 DataFrame 이름을 사용하는가': False,
    '새 그래프가 선택한 질문에 답하는가': False,
    '축과 단위를 설명할 수 있는가': False,
    '관찰을 원인으로 단정하지 않았는가': False,
})

In [ ]:
codex_check

### 발표 준비

선택한 전공 자료의 질문을 먼저 읽고, 표에서 숫자 하나, 그래프에서 특징 하나를 가리킵니다. 환경·GIS 경로를 선택했다면 지도에서 구·군 하나도 함께 가리킵니다. 마지막에는 이 자료만으로 판단할 수 없는 내용을 말합니다.

## 제출 전 확인

- 분석 질문이 한 문장으로 적혀 있는가?
- 선택한 전공 자료의 결과 표를 확인했는가?
- 그래프의 두 축을 설명할 수 있는가?
- GIS 경로에서는 지도 점의 색을 설명할 수 있는가?
- 결과 두 문장과 한계 한 문장을 작성했는가?